# 40 - Build Silver Jobs

Materializes Job Cost master data, dates, people, financial snapshots, billing transactions, and material costs.

In [1]:
# Attach the schema-enabled Lakehouse containing the Bronze shortcuts and
# make it the default Lakehouse before running this notebook.
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
METADATA_SCHEMA = "metadata"
COMPANY_KEY = "NAC"  # Change when processing another GP company database.

def q(identifier: str) -> str:
    return f"`{identifier.replace('`', '``')}`"

def sql_string(value: str) -> str:
    return "'" + value.replace("'", "''") + "'"

def table_exists(schema_name: str, table_name: str) -> bool:
    try:
        return spark.catalog.tableExists(f"{schema_name}.{table_name}")
    except Exception:
        return False

def require_tables(table_names: list[str], schema_name: str = BRONZE_SCHEMA) -> None:
    missing = [name for name in table_names if not table_exists(schema_name, name)]
    if missing:
        raise RuntimeError(
            f"Missing tables in {schema_name}: {', '.join(missing)}. "
            "Confirm the OneLake shortcuts and exact table names."
        )

company = sql_string(COMPANY_KEY)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {q(SILVER_SCHEMA)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {q(GOLD_SCHEMA)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {q(METADATA_SCHEMA)}")
print(f"Company: {COMPANY_KEY}; Bronze schema: {BRONZE_SCHEMA}")

StatementMeta(, c62403e7-a76b-4e8d-a186-b174827bfc53, 3, Finished, Available, Finished, False)

Company: NAC; Bronze schema: bronze


In [2]:
require_tables(["JC00102", "JC30001", "JC00107", "JC20001", "JC20501"])

StatementMeta(, c62403e7-a76b-4e8d-a186-b174827bfc53, 4, Finished, Available, Finished, False)

## Job user-defined fields and master

In [3]:
spark.sql(f"""
CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.job_udf
USING DELTA
AS
SELECT
    {company} AS company_key,
    TRIM(WS_Job_Number) AS job_number,
    TRY_CAST(NULLIF(TRIM(WS_Manager_ID_2), '') AS BIGINT) AS superintendent_employee_code,
    TRY_CAST(NULLIF(TRIM(WS_Engineer), '') AS BIGINT) AS engineer_employee_code,
    TRY_CAST(NULLIF(TRIM(User_Define_1), '') AS DOUBLE) AS estimated_overtime_hours,
    TRY_CAST(NULLIF(TRIM(User_Define_2), '') AS INT) AS estimated_manpower,
    TRY_CAST(NULLIF(TRIM(User_Define_3), '') AS BIGINT) AS foreman_employee_code,
    TRY_CAST(NULLIF(TRIM(User_Define_4), '') AS INT) AS material_analysis_code,
    TRY_CAST(NULLIF(TRIM(User_Define_5), '') AS DOUBLE) AS projected_hours,
    NULLIF(TRIM(User_Define_6), '') AS job_scope,
    TRY_CAST(NULLIF(TRIM(USRDAT01), '') AS DATE) AS salesforce_booking_date,
    CASE WHEN User_Defined_CB_1 = 1 THEN TRUE ELSE FALSE END AS permit_required_flag,
    CASE WHEN User_Defined_CB_2 = 1 THEN TRUE ELSE FALSE END AS priority_flag,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(BRONZE_SCHEMA)}.JC00107
WHERE TRIM(WS_Job_Number) <> ''
""")

from pyspark.sql import functions as F
job_source_df = (
    spark.table(f"{BRONZE_SCHEMA}.JC00102")
        .withColumn("source_priority", F.lit(1))
        .withColumn("source_status", F.lit("Current"))
        .unionByName(
            spark.table(f"{BRONZE_SCHEMA}.JC30001")
                .withColumn("source_priority", F.lit(2))
                .withColumn("source_status", F.lit("History")),
            allowMissingColumns=True
        )
)
job_source_df.createOrReplaceTempView("_job_source")

spark.sql(f"""
CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.job
USING DELTA
AS
WITH ranked AS (
    SELECT *, ROW_NUMBER() OVER (
        PARTITION BY TRIM(WS_Job_Number)
        ORDER BY source_priority
    ) AS rn
    FROM _job_source
    WHERE TRIM(WS_Job_Number) <> ''
)
SELECT
    {company} AS company_key,
    TRIM(j.WS_Job_Number) AS job_number,
    TRIM(j.Divisions) AS division_code,
    CASE
        WHEN j.Divisions LIKE '%_HVAC_%' THEN 'Service'
        WHEN j.Divisions LIKE '%_SP_%' THEN 'Special Projects'
    END AS division_name,
    CASE
        WHEN j.Divisions LIKE '%_HVAC_%' THEN 'dispatch@nacgroup.com'
        WHEN j.Divisions LIKE '%_SP_%' THEN 'sp_operations@nacgroup.com'
    END AS division_email,
    TRIM(j.CUSTNMBR) AS customer_code,
    TRIM(j.Job_Address_Code) AS location_code,
    TRIM(j.WS_Job_Name) AS job_name,
    TRIM(j.User_Define_1) AS job_description,
    TRY_CAST(j.Calc_Pct_Compl_to_Date AS INT) AS percent_complete,
    CASE
        WHEN j.source_status = 'History' THEN 'Closed'
        WHEN j.WS_Inactive = 1 THEN 'Inactive'
        ELSE 'Open'
    END AS job_status,
    u.job_scope,
    CONCAT(TRIM(j.Job_Address_Code), TRIM(j.CUSTNMBR)) AS location_xref,
    CURRENT_TIMESTAMP() AS _processed_at
FROM ranked j
LEFT JOIN {q(SILVER_SCHEMA)}.job_udf u
  ON {company} = u.company_key AND TRIM(j.WS_Job_Number) = u.job_number
WHERE j.rn = 1
""")

StatementMeta(, c62403e7-a76b-4e8d-a186-b174827bfc53, 5, Finished, Available, Finished, False)

DataFrame[]

## Job dates and people

In [4]:
spark.sql(f"""
CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.job_date
USING DELTA
AS
WITH dates AS (
    SELECT 1 AS source_priority, TRIM(WS_Job_Number) AS job_number,
           CAST(Schedule_Start_Date AS DATE) AS scheduled_start_date,
           CAST(Sched_Completion_Date AS DATE) AS scheduled_completion_date,
           CAST(ACTCOMPDATE AS DATE) AS actual_completion_date
    FROM {q(BRONZE_SCHEMA)}.JC00102
    UNION ALL
    SELECT 2, TRIM(WS_Job_Number), CAST(Schedule_Start_Date AS DATE),
           CAST(Sched_Completion_Date AS DATE), CAST(ACTCOMPDATE AS DATE)
    FROM {q(BRONZE_SCHEMA)}.JC30001
), ranked AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY job_number ORDER BY source_priority) AS rn
    FROM dates
)
SELECT
    {company} AS company_key,
    job_number,
    CASE WHEN scheduled_start_date > DATE'1900-01-01' THEN scheduled_start_date END AS scheduled_start_date,
    CASE WHEN scheduled_completion_date > DATE'1900-01-01' THEN scheduled_completion_date END AS scheduled_completion_date,
    CASE WHEN actual_completion_date > DATE'1900-01-01' THEN actual_completion_date END AS actual_completion_date,
    CURRENT_TIMESTAMP() AS _processed_at
FROM ranked WHERE rn = 1
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.job_people
USING DELTA
AS
SELECT
    j.company_key,
    j.job_number,
    TRY_CAST(NULLIF(TRIM(p.WS_Manager_ID), '') AS BIGINT) AS project_manager_employee_code,
    TRY_CAST(NULLIF(TRIM(p.Estimator_ID), '') AS BIGINT) AS sales_rep_employee_code,
    u.foreman_employee_code,
    u.superintendent_employee_code,
    u.engineer_employee_code,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(SILVER_SCHEMA)}.job j
LEFT JOIN {q(BRONZE_SCHEMA)}.JC00102 p
  ON j.job_number = TRIM(p.WS_Job_Number)
LEFT JOIN {q(SILVER_SCHEMA)}.job_udf u
  ON j.company_key = u.company_key AND j.job_number = u.job_number
""")

StatementMeta(, c62403e7-a76b-4e8d-a186-b174827bfc53, 6, Finished, Available, Finished, False)

DataFrame[]

## Job financial snapshot

In [5]:
spark.sql(f"""
CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.job_financial_snapshot
USING DELTA
AS
WITH ranked AS (
    SELECT *, ROW_NUMBER() OVER (
        PARTITION BY TRIM(WS_Job_Number) ORDER BY source_priority
    ) AS rn
    FROM _job_source
)
SELECT
    {company} AS company_key,
    TRIM(WS_Job_Number) AS job_number,
    CAST(Expected_Contract AS DECIMAL(18,2)) AS expected_contract,
    TRY_CAST(Calc_Pct_Compl_to_Date AS INT) AS percent_complete,
    CAST(Est_Equipment_Cost AS DECIMAL(18,2)) AS estimated_equipment,
    CAST(Est_Material_Cost AS DECIMAL(18,2)) AS estimated_material,
    CAST(Est_Cost_UserDef1 AS DECIMAL(18,2)) AS estimated_subcontractor,
    CAST(Est_Labor_Cost AS DECIMAL(18,2)) AS estimated_labor,
    CAST(Est_Misc_Other_Cost AS DECIMAL(18,2)) AS estimated_miscellaneous,
    CAST(Est_Cost_UserDef2 AS DECIMAL(18,2)) AS estimated_burden,
    CAST(Est_Cost_UserDef3 AS DECIMAL(18,2)) AS estimated_contingency,
    CAST(Est_Cost_UserDef4 AS DECIMAL(18,2)) AS estimated_warranty,
    CAST(Total_Estimated_Cost AS DECIMAL(18,2)) AS total_estimated_cost,
    CAST(Act_Equipment_Cost_TTD AS DECIMAL(18,2)) AS actual_equipment,
    CAST(Act_Materials_Cost_TTD AS DECIMAL(18,2)) AS actual_material,
    CAST(Act_Cost_TTD_UserDef1 AS DECIMAL(18,2)) AS actual_subcontractor,
    CAST(Act_Labor_Cost_TTD AS DECIMAL(18,2)) AS actual_labor,
    CAST(Act_Cost_TTD_UserDef2 AS DECIMAL(18,2)) AS actual_burden,
    CAST(Act_Misc_Other_Cost_TTD AS DECIMAL(18,2)) AS actual_miscellaneous,
    CAST(Act_Cost_TTD_UserDef4 AS DECIMAL(18,2)) AS actual_warranty,
    CAST(Total_Actual_Cost AS DECIMAL(18,2)) AS total_actual_cost,
    CAST(Committed_Equipment_TTD AS DECIMAL(18,2)) AS committed_equipment,
    CAST(Committed_Materials_TTD AS DECIMAL(18,2)) AS committed_material,
    CAST(Committed_TTD_UserDef1 AS DECIMAL(18,2)) AS committed_subcontractor,
    CAST(Committed_Labor_TTD AS DECIMAL(18,2)) AS committed_labor,
    CAST(Committed_Subs_TTD AS DECIMAL(18,2)) AS committed_rental,
    CAST(Committed_Other_TTD AS DECIMAL(18,2)) AS committed_other,
    CAST(Committed_Cost AS DECIMAL(18,2)) AS total_committed_cost,
    CAST(Revsd_Equip_Forecasted AS DECIMAL(18,2)) AS forecast_equipment,
    CAST(Revsd_Materials_Forecst AS DECIMAL(18,2)) AS forecast_material,
    CAST(Revsd_Forecast_UserDef1 AS DECIMAL(18,2)) AS forecast_subcontractor,
    CAST(Revsd_Labor_Forecasted AS DECIMAL(18,2)) AS forecast_labor,
    CAST(Revsd_Other_Forecst_Cst AS DECIMAL(18,2)) AS forecast_other,
    CAST(Revsd_Forecast_UserDef2 AS DECIMAL(18,2)) AS forecast_burden,
    CAST(Revsd_Forecast_UserDef3 AS DECIMAL(18,2)) AS forecast_contingency,
    CAST(Revsd_Forecast_UserDef4 AS DECIMAL(18,2)) AS forecast_warranty,
    CAST(Tot_Revsd_Forecast_Cost AS DECIMAL(18,2)) AS total_forecast_cost,
    CAST(Est_Labor_Units_TTD AS DOUBLE) AS estimated_hours,
    CAST(Act_Labor_Units_TTD AS DOUBLE) AS actual_hours,
    CURRENT_DATE() AS snapshot_date,
    CURRENT_TIMESTAMP() AS _processed_at
FROM ranked WHERE rn = 1 AND TRIM(WS_Job_Number) <> ''
""")

StatementMeta(, c62403e7-a76b-4e8d-a186-b174827bfc53, 7, Finished, Available, Finished, False)

DataFrame[]

## Job cost and billing transactions

In [6]:
spark.sql(f"""
CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.job_cost
USING DELTA
AS
SELECT
    {company} AS company_key,
    TRIM(VENDORID) AS vendor_code,
    TRIM(WS_Job_Number) AS job_number,
    TRY_CAST(JRNENTRY AS BIGINT) AS journal_number,
    TRY_CAST(Cost_Element AS INT) AS cost_code,
    TRIM(Cost_Code_Description) AS cost_label,
    TRY_CAST(DOCTYPE AS INT) AS document_code,
    CAST(DOCDATE AS DATE) AS document_date,
    CAST(DOCAMNT AS DECIMAL(18,2)) AS document_amount,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(BRONZE_SCHEMA)}.JC20001
WHERE (UPPER(TRIM(Cost_Code_Description)) = 'MATERIALS' OR Cost_Element = 1)
  AND COALESCE(DOCAMNT, 0) <> 0
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.job_billing
USING DELTA
AS
SELECT
    {company} AS company_key,
    TRIM(WS_Job_Number) AS job_number,
    TRIM(Docnumbr) AS document_number,
    CAST(DOCDATE AS DATE) AS document_date,
    TRIM(CUSTNMBR) AS customer_code,
    CAST(DOCAMNT AS DECIMAL(18,2)) AS document_amount,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(BRONZE_SCHEMA)}.JC20501
WHERE TRIM(WS_Job_Number) <> ''
""")

StatementMeta(, c62403e7-a76b-4e8d-a186-b174827bfc53, 8, Finished, Available, Finished, False)

DataFrame[]

## Review

In [7]:
for name in ["job_udf", "job", "job_date", "job_people",
             "job_financial_snapshot", "job_cost", "job_billing"]:
    print(f"{SILVER_SCHEMA}.{name}: {spark.table(f'{SILVER_SCHEMA}.{name}').count():,} rows")

StatementMeta(, c62403e7-a76b-4e8d-a186-b174827bfc53, 9, Finished, Available, Finished, False)

silver.job_udf: 650 rows
silver.job: 4,480 rows
silver.job_date: 4,480 rows
silver.job_people: 4,480 rows
silver.job_financial_snapshot: 4,480 rows
silver.job_cost: 972 rows
silver.job_billing: 686 rows
